In [1]:
import tensorflow as tf
import numpy as np
import pandas as pd
import os
import glob
from PIL import Image

# ==========================================
# 1. CẤU HÌNH ĐƯỜNG DẪN
# ==========================================
BASE_PATH = "C:/DUT/Ki 2/Edge AI/dataset/"
TEST_DIR = os.path.join(BASE_PATH, "test")
TFLITE_MODEL_PATH = "best_model.tflite"
SUBMISSION_PATH = "submission.csv"

IMG_HEIGHT = 32
IMG_WIDTH = 32

# ==========================================
# 2. TẢI MÔ HÌNH TFLITE (INT8)
# ==========================================
print("🔄 Đang tải mô hình TFLite...")
interpreter = tf.lite.Interpreter(model_path=TFLITE_MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Lấy thông số quantization (Scale & Zero Point)
input_scale, input_zero_point = input_details[0]['quantization']

# ==========================================
# 3. CHUẨN BỊ DANH SÁCH ẢNH TEST
# ==========================================
# Tìm tất cả file ảnh trong thư mục test
test_images = glob.glob(os.path.join(TEST_DIR, "*.[jp][pn]*"))
# QUAN TRỌNG: Sắp xếp theo tên file để Id khớp với thứ tự (00000, 00001,...)
test_images.sort()

print(f"🔎 Tìm thấy {len(test_images)} ảnh trong tập test.")

results = []

# ==========================================
# 4. CHẠY DỰ ĐOÁN (INFERENCE)
# ==========================================
print("🚀 Bắt đầu dự đoán nhãn...")

for img_path in test_images:
    # Lấy Id từ tên file (Ví dụ: "00005.png" -> "00005")
    img_id = os.path.basename(img_path).split('.')[0]
    
    # Tiền xử lý ảnh giống hệt lúc huấn luyện
    img = Image.open(img_path).convert('RGB').resize((IMG_WIDTH, IMG_HEIGHT))
    img_array = np.array(img, dtype=np.float32) / 255.0
    
    # Chuyển đổi sang INT8 (Yêu cầu bắt buộc của mô hình lượng tử hóa)
    if input_scale != 0:
        img_array = (img_array / input_scale) + input_zero_point
    img_input = np.expand_dims(img_array.astype(np.int8), axis=0)

    # Đưa vào Interpreter
    interpreter.set_tensor(input_details[0]['index'], img_input)
    interpreter.invoke()
    
    # Lấy kết quả đầu ra
    output_data = interpreter.get_tensor(output_details[0]['index'])[0]
    # predicted_label chính là vị trí có giá trị lớn nhất (Argmax)
    predicted_label = np.argmax(output_data)
    
    results.append([img_id, predicted_label])

# ==========================================
# 5. XUẤT FILE SUBMISSION.CSV
# ==========================================
submission_df = pd.DataFrame(results, columns=['Id', 'Label'])

# Đảm bảo cột Id được định dạng đúng (giữ các số 0 ở đầu nếu BTC yêu cầu chuỗi)
# Nếu BTC yêu cầu Id là số (0, 1, 2...) thì dùng: submission_df['Id'] = submission_df['Id'].astype(int)
# Ở đây mình giữ nguyên theo tên file (00000, 00001...)

submission_df.to_csv(SUBMISSION_PATH, index=False)

print("\n" + "="*40)
print(f"✅ Đã xuất file thành công: {SUBMISSION_PATH}")
print(f"📊 Tổng số dòng: {len(submission_df)}")
print("="*40)

# Hiển thị 5 dòng đầu để kiểm tra
print(submission_df.head())

🔄 Đang tải mô hình TFLite...
🔎 Tìm thấy 1000 ảnh trong tập test.
🚀 Bắt đầu dự đoán nhãn...


c:\Users\ngong\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



✅ Đã xuất file thành công: submission.csv
📊 Tổng số dòng: 1000
      Id  Label
0  00000      8
1  00007      2
2  00012      2
3  00014      8
4  00022      1
